# 128-DWT（3線形補間）カリキュラム事前学習

比較対象の通常128モデルと出力解像度を揃える。処理は `128×256×256 → 3線形補間 → 64×128×128 → 3D Haar DWT → 8成分 (32×64×64) → 既存ネットワーク内の4群化 → 全8成分を同一DVFでwarp → inverse DWT → 64×128×128`。

カリキュラムの ±1〜±40 px は、**再構成後の `64×128×128` 画像のピクセル単位**である。係数格子は各軸半分のため、DVF教師値は係数格子へ落とす際に `/2` している。

In [ ]:
from pathlib import Path
import sys
import torch

# ノートブックを Saito フォルダから実行する想定。voxelmorph の親を import path に追加する。
PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'voxelmorph').is_dir():
    raise FileNotFoundError(f'voxelmorph folder was not found under: {PROJECT_ROOT}')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dwt128_trilinear_curriculum import TARGET_SHAPE, COEFFICIENT_SHAPE, run_curriculum_pretraining

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DATA_PATH = Path('Data/TrainData_NoBed.npz')
CHECKPOINT_DIR = Path('128dwt_trilinear_curriculum_checkpoints')

print('device:', device)
print('target image:', TARGET_SHAPE)
print('DWT coefficients:', COEFFICIENT_SHAPE)
print('training data:', DATA_PATH.resolve())
print('checkpoints:', CHECKPOINT_DIR.resolve())

In [ ]:
# 256版と同じカリキュラム: 1 stage = 2,000 epoch、1→40 px、計80,000 epoch。
TOTAL_EPOCHS = 80_000
STAGE_EPOCHS = 2_000
RUN_TRAINING = False  # 実行前に True へ変更

# ここは事前学習なので None のまま。
# 別患者fine-tune後に同じ ±1→±40 学習を追加で行う場合だけ、
# その fine-tune 最終重みへのパスを指定し、prefix/output_dir を別名にする。
INITIAL_CHECKPOINT = None
CHECKPOINT_PREFIX = '128dwt_trilinear_curriculum'

if RUN_TRAINING:
    model_dwt, final_path, history = run_curriculum_pretraining(
        data_path=DATA_PATH,
        output_dir=CHECKPOINT_DIR,
        total_epochs=TOTAL_EPOCHS,
        stage_epochs=STAGE_EPOCHS,
        batch_size=2,
        initial_checkpoint=INITIAL_CHECKPOINT,
        checkpoint_prefix=CHECKPOINT_PREFIX,
        device=device,
    )
    print('final checkpoint:', final_path.resolve())
else:
    print('RUN_TRAINING=False: 設定確認のみ。True にしてから実行する。')

## 保存される重み

- 各段階: `128dwt_trilinear_curriculum_stage_01_epoch_02000.pth` 〜 `...stage_40_epoch_80000.pth`
- カリキュラム最終: `128dwt_trilinear_curriculum_1to40_final.pth`

次に作る別患者・肺領域優先fine-tuneは、この最終重みを入力にし、別フォルダへ `128dwt_finetuned_different_patients_final.pth` を保存する。その後の ±1〜±40 px 学習もさらに別名で保存するので、最終的に **fine-tune直後** と **fine-tune後の1→40 px学習後** の2重みを同じテストデータで比較できる。